# Simulation Replay Driver

Driver notebook for replaying recorded sessions through the refactored Branch 1 RD-patch + Branch 3 U-Net navigation stack.

This notebook keeps program state in profile objects and calls the current repository code directly. It does not modify legacy notebooks or rewrite replay scripts.

Code is expected to come from `gs://miamioh-resa-data/CapstoneData/code_v2/RESA_mmWave/` when running in Colab. Local runs use the current repo checkout.

## Flow

1. Detect Colab vs local runtime.
2. Configure GCS helpers and optionally sync the refactored code tree.
3. Define a frozen replay profile separate from live execution.
4. Pull a rolling batch of replay sessions when running in Colab.
5. Inject the profile into the canonical replay config shape.
6. Run `sim_eval/branch3_replay_simulation.py` in-process.
7. Optionally run OneFormer decision evaluation and overlay diagnostics.
8. Upload artifacts back to GCS.

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import textwrap
import uuid
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "branch2" / "navigation_config.py").exists() and (candidate / "sim_eval").exists():
            return candidate
    return None

if IN_COLAB:
    WORK_ROOT = Path("/content/work")
    CODE_ROOT = WORK_ROOT / "code"
    LOCAL_DATA_ROOT = WORK_ROOT / "data"
    LOCAL_OUT_ROOT = WORK_ROOT / "outputs" / "simulation_replay"
else:
    detected_root = find_repo_root(Path.cwd().resolve())
    if detected_root is None:
        raise RuntimeError("Run this notebook from the RESA_mmWave repo root, or edit CODE_ROOT manually.")
    WORK_ROOT = detected_root
    CODE_ROOT = detected_root
    LOCAL_DATA_ROOT = CODE_ROOT / "LLM_ML" / "data"
    LOCAL_OUT_ROOT = CODE_ROOT / "outputs" / "simulation_replay"

for path in (WORK_ROOT, CODE_ROOT, LOCAL_DATA_ROOT, LOCAL_OUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("CODE_ROOT:", CODE_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("LOCAL_OUT_ROOT:", LOCAL_OUT_ROOT)

In [ ]:
PROJECT_ID = "fluent-webbing-496616-u8"
BUCKET_NAME = "miamioh-resa-data"
GCS_MOUNT_POINT = Path("/content/gcs")
GCS_CODE_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/code_v2/RESA_mmWave/"
GCS_OUTPUT_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/intermediate/replay_eval/runs"
GCS_PUBLISHED_OUTPUT_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/published/replay_eval/runs"
GCS_CATALOG_URI = f"gs://{BUCKET_NAME}/CapstoneData/curated/manifests/session_catalog.csv"
GCS_CURATED_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/curated/sessions/gold"
GCS_SYNC_ROOT = None

GCLOUD_PROCESS_COUNT = 4
GCLOUD_THREAD_COUNT = 16

def run(cmd, *, cwd: Path | None = None, check: bool = True, capture: bool = False):
    if isinstance(cmd, str):
        printable = cmd
        use_shell = True
    else:
        printable = " ".join(str(x) for x in cmd)
        use_shell = False
    print("$", printable)
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=use_shell,
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if capture:
        return result.stdout or ""
    return result

def configure_gcloud():
    if not IN_COLAB:
        print("Local runtime: skipping Colab auth and GCS mount setup.")
        return
    from google.colab import auth

    auth.authenticate_user()
    if shutil.which("gcloud") is None:
        run("echo 'deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main' | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list")
        run("curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key --keyring /usr/share/keyrings/cloud.google.gpg add -")
        run(["sudo", "apt-get", "update", "-qq"])
        run(["sudo", "apt-get", "install", "-y", "-qq", "google-cloud-cli"])

    run(["gcloud", "config", "set", "project", PROJECT_ID])
    run(["gcloud", "config", "set", "storage/process_count", str(GCLOUD_PROCESS_COUNT)])
    run(["gcloud", "config", "set", "storage/thread_count", str(GCLOUD_THREAD_COUNT)])

    if shutil.which("gcsfuse") is None:
        run("export GCSFUSE_REPO=gcsfuse-$(lsb_release -c -s); echo deb https://packages.cloud.google.com/apt $GCSFUSE_REPO main | sudo tee /etc/apt/sources.list.d/gcsfuse.list")
        run("curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -")
        run(["sudo", "apt-get", "update", "-qq"])
        run(["sudo", "apt-get", "install", "-y", "-qq", "gcsfuse"])

    GCS_MOUNT_POINT.mkdir(parents=True, exist_ok=True)
    run(["umount", str(GCS_MOUNT_POINT)], check=False)
    run(["gcsfuse", "--implicit-dirs", BUCKET_NAME, str(GCS_MOUNT_POINT)], check=False)
    print("Mounted path candidate:", GCS_MOUNT_POINT / "CapstoneData")

configure_gcloud()


In [ ]:
def is_gs_uri(value) -> bool:
    return str(value).startswith("gs://")

def path_to_gs_uri(path) -> str:
    s = str(path)
    if s.startswith("gs://"):
        return s
    mount_prefix = str(GCS_MOUNT_POINT / BUCKET_NAME)
    if s.startswith(mount_prefix):
        return "gs://" + s[len(str(GCS_MOUNT_POINT)) + 1:].strip("/")
    capstone_prefix = str(GCS_MOUNT_POINT / "CapstoneData")
    if s.startswith(capstone_prefix):
        return f"gs://{BUCKET_NAME}/CapstoneData/" + s[len(capstone_prefix):].strip("/")
    return s

def gcloud_storage_rsync(src, dst, *, delete: bool = False, excludes: list[str] | None = None):
    cmd = ["gcloud", "storage", "rsync", "--recursive"]
    if delete:
        cmd.append("--delete-unmatched-destination-objects")
    for pattern in excludes or []:
        cmd.extend(["--exclude", pattern])
    cmd.extend([path_to_gs_uri(src), path_to_gs_uri(dst)])
    return run(cmd)

def gcloud_storage_cp(src, dst, *, recursive: bool = False):
    cmd = ["gcloud", "storage", "cp"]
    if recursive:
        cmd.append("--recursive")
    cmd.extend([path_to_gs_uri(src), path_to_gs_uri(dst)])
    return run(cmd)

def gcloud_storage_ls(uri: str) -> list[str]:
    if shutil.which("gcloud") is None:
        return []
    out = run(["gcloud", "storage", "ls", uri], check=False, capture=True)
    return [line.strip() for line in out.splitlines() if line.strip().startswith("gs://")]

def first_existing(*paths: Path) -> Path | None:
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None

In [ ]:
if IN_COLAB:
    CODE_ROOT.mkdir(parents=True, exist_ok=True)
    gcloud_storage_rsync(
        GCS_CODE_ROOT,
        CODE_ROOT,
        excludes=[
            r"LLM_ML/data/.*",
            r"data/.*",
            r"outputs/.*",
            r"__pycache__/.*",
            r"\.ipynb_checkpoints/.*",
        ],
    )
else:
    print("Local runtime: using current checkout, no code sync.")

required_repo_files = [
    CODE_ROOT / "sim_eval" / "branch3_replay_simulation.py",
    CODE_ROOT / "sim_eval" / "evaluate_replay_decisions_against_oneformer.py",
    CODE_ROOT / "sim_eval" / "visualize_seg_radar_overlay.py",
    CODE_ROOT / "branch2" / "navigation_config.py",
    CODE_ROOT / "branch3" / "branch3_unet.py",
    CODE_ROOT / "branch1" / "models" / "hybrid_rd" / "hybrid_rd_model.py",
]
missing = [str(path) for path in required_repo_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing expected refactored repo files:\n" + "\n".join(missing))
print("Repo layout verified.")

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "tqdm", "opencv-python", "matplotlib", "scipy", "scikit-learn"])

if importlib.util.find_spec("torch") is None:
    run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])
else:
    import torch
    print("torch:", torch.__version__)

In [ ]:
@dataclass(frozen=True)
class ReplayStorageProfile:
    gcs_code_root: str
    gcs_catalog_uri: str
    gcs_curated_root: str
    gcs_sync_root: str | None
    gcs_output_root: str
    gcs_published_output_root: str
    local_repo_root: Path
    local_data_root: Path
    local_output_root: Path

@dataclass(frozen=True)
class ReplayRunProfile:
    run_id: str
    processing_root: Path
    sync_root: Path | None
    sessions_dir: Path
    output_dir: Path
    branch1_checkpoint: Path
    branch3_unet_checkpoint: Path
    mmwave_cfg_path: Path
    extrinsics_json: Path
    max_sessions: int
    session_selection: str
    session_stride: int
    session_seed: int
    session_names: str | None
    catalog_quality_tiers: str
    catalog_split_allowlist: str | None
    promote_to_published: bool
    ui_style: str
    fps: float
    width: int
    height: int
    no_video: bool
    no_color_video: bool
    debug_frame_snapshots: bool
    debug_frame_snapshot_stride: int
    export_frame_predictions: bool
    export_point_predictions: bool
    run_oneformer_eval: bool
    run_overlay_diagnostics: bool
    catalog_local_path: Path

ROLLING_REPLAY_ROOT = LOCAL_DATA_ROOT / "_rolling_sim_replay_sessions"
LOCAL_CURATED_ROOT = LOCAL_DATA_ROOT / "curated_sessions_gold"
LOCAL_CATALOG_PATH = LOCAL_OUT_ROOT / "session_catalog.csv"

storage_profile = ReplayStorageProfile(
    gcs_code_root=GCS_CODE_ROOT,
    gcs_catalog_uri=GCS_CATALOG_URI,
    gcs_curated_root=GCS_CURATED_ROOT,
    gcs_sync_root=GCS_SYNC_ROOT,
    gcs_output_root=GCS_OUTPUT_ROOT,
    gcs_published_output_root=GCS_PUBLISHED_OUTPUT_ROOT,
    local_repo_root=CODE_ROOT,
    local_data_root=LOCAL_DATA_ROOT,
    local_output_root=LOCAL_OUT_ROOT,
)

profile = ReplayRunProfile(
    run_id=f"sim_replay_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}",
    processing_root=ROLLING_REPLAY_ROOT if IN_COLAB else LOCAL_CURATED_ROOT,
    sync_root=None if IN_COLAB else None,
    sessions_dir=ROLLING_REPLAY_ROOT if IN_COLAB else LOCAL_CURATED_ROOT,
    output_dir=LOCAL_OUT_ROOT,
    branch1_checkpoint=first_existing(
        CODE_ROOT / "published" / "branch1" / "checkpoints" / "3branch_best_model.pt",
        CODE_ROOT / "models" / "3branch_best_model.pt",
        CODE_ROOT / "models" / "hybrid_rd" / "3branch_best_model.pt",
    ) or CODE_ROOT / "published" / "branch1" / "checkpoints" / "3branch_best_model.pt",
    branch3_unet_checkpoint=first_existing(
        CODE_ROOT / "published" / "branch3" / "checkpoints" / "unet_best_model.pt",
        CODE_ROOT / "models" / "unet_best_model.pt",
        CODE_ROOT / "branch3" / "models" / "unet_best_model.pt",
    ) or CODE_ROOT / "published" / "branch3" / "checkpoints" / "unet_best_model.pt",
    mmwave_cfg_path=CODE_ROOT / "config" / "profile_objdet.cfg",
    extrinsics_json=CODE_ROOT / "config" / "radar_camera_extrinsics.json",
    max_sessions=5,
    session_selection="sequential",
    session_stride=5,
    session_seed=42,
    session_names=None,
    catalog_quality_tiers="gold",
    catalog_split_allowlist=None,
    promote_to_published=False,
    ui_style="integrated",
    fps=10.0,
    width=1280,
    height=720,
    no_video=False,
    no_color_video=False,
    debug_frame_snapshots=True,
    debug_frame_snapshot_stride=10,
    export_frame_predictions=True,
    export_point_predictions=True,
    run_oneformer_eval=False,
    run_overlay_diagnostics=False,
    catalog_local_path=LOCAL_CATALOG_PATH,
)

profile.output_dir.mkdir(parents=True, exist_ok=True)
profile_path = profile.output_dir / f"{profile.run_id}_profile.json"
profile_path.write_text(json.dumps({
    "storage_profile": {k: str(v) for k, v in asdict(storage_profile).items()},
    "run_profile": {k: str(v) for k, v in asdict(profile).items()},
}, indent=2), encoding="utf-8")
print(profile_path)
print(json.dumps({k: str(v) for k, v in asdict(profile).items()}, indent=2))


In [ ]:
import tarfile

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from tools.session_selector import load_catalog, select_sessions, sync_catalog_from_gcs


def parse_csv_list(raw: str | None) -> tuple[str, ...]:
    if raw is None:
        return ()
    return tuple(part.strip() for part in str(raw).split(",") if part and part.strip())


def sync_local_catalog(profile: ReplayRunProfile, storage: ReplayStorageProfile) -> Path:
    if IN_COLAB:
        return sync_catalog_from_gcs(profile.catalog_local_path, storage.gcs_catalog_uri)
    profile.catalog_local_path.parent.mkdir(parents=True, exist_ok=True)
    fallback = CODE_ROOT / "manifests" / "session_catalog.csv"
    if not profile.catalog_local_path.exists() and fallback.exists():
        shutil.copy2(fallback, profile.catalog_local_path)
    return profile.catalog_local_path


def load_replay_catalog(profile: ReplayRunProfile, storage: ReplayStorageProfile) -> pd.DataFrame:
    local_catalog = sync_local_catalog(profile, storage)
    df = load_catalog(local_catalog)
    quality_tiers = parse_csv_list(profile.catalog_quality_tiers) or ("gold",)
    rows = select_sessions(df, quality_tier=quality_tiers, split_exclude=("none", ""))
    allowed_splits = set(parse_csv_list(profile.catalog_split_allowlist))
    if allowed_splits:
        rows = rows[rows["split"].astype(str).isin(allowed_splits)].reset_index(drop=True)
    if profile.session_names:
        requested = [x.strip() for x in profile.session_names.split(",") if x.strip()]
        available = set(rows["session_id"].astype(str).tolist())
        missing = [name for name in requested if name not in available]
        if missing:
            raise FileNotFoundError("Requested sessions not found in curated catalog: " + ", ".join(missing))
        rows = rows[rows["session_id"].astype(str).isin(requested)].copy()
    if rows.empty:
        raise RuntimeError("No curated replay sessions matched the current catalog filters.")
    return rows.sort_values(["dataset_id", "session_id"]).reset_index(drop=True)


def select_session_names(names: list[str], profile: ReplayRunProfile) -> list[str]:
    if profile.session_names:
        requested = [x.strip() for x in profile.session_names.split(",") if x.strip()]
        available = set(names)
        missing = [name for name in requested if name not in available]
        if missing:
            raise FileNotFoundError("Requested sessions not found: " + ", ".join(missing))
        return requested

    selected = list(names)
    if profile.session_selection == "stride":
        selected = selected[::max(1, profile.session_stride)]
    elif profile.session_selection == "random":
        rng = random.Random(profile.session_seed)
        rng.shuffle(selected)
    elif profile.session_selection != "sequential":
        raise ValueError(f"Unsupported session_selection={profile.session_selection!r}")

    if profile.max_sessions and profile.max_sessions > 0:
        selected = selected[:profile.max_sessions]
    return selected


def clear_rolling_replay_dirs():
    if ROLLING_REPLAY_ROOT.exists():
        shutil.rmtree(ROLLING_REPLAY_ROOT)
    ROLLING_REPLAY_ROOT.mkdir(parents=True, exist_ok=True)


def extract_session_archive(archive_path: Path, dst_root: Path, expected_session_name: str) -> Path:
    tmp_root = dst_root / "_extract_tmp" / expected_session_name
    shutil.rmtree(tmp_root, ignore_errors=True)
    tmp_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as tf:
        base = tmp_root.resolve()
        for member in tf.getmembers():
            target = (tmp_root / member.name).resolve()
            if not str(target).startswith(str(base)):
                raise RuntimeError(f"Unsafe archive path in {archive_path}: {member.name}")
        tf.extractall(tmp_root)
    direct = tmp_root / expected_session_name
    final_session = dst_root / expected_session_name
    shutil.rmtree(final_session, ignore_errors=True)
    if direct.exists() and direct.is_dir():
        shutil.move(str(direct), str(final_session))
    else:
        final_session.mkdir(parents=True, exist_ok=True)
        for item in sorted(tmp_root.iterdir()):
            shutil.move(str(item), str(final_session / item.name))
    shutil.rmtree(dst_root / "_extract_tmp", ignore_errors=True)
    return final_session


def download_rolling_sessions(profile: ReplayRunProfile, storage: ReplayStorageProfile) -> tuple[list[str], pd.DataFrame]:
    catalog_rows = load_replay_catalog(profile, storage)
    selected_names = select_session_names(catalog_rows["session_id"].astype(str).tolist(), profile)
    selected_rows = catalog_rows[catalog_rows["session_id"].astype(str).isin(selected_names)].copy()
    selected_rows = selected_rows.set_index("session_id").loc[selected_names].reset_index()

    if not IN_COLAB:
        local_sessions = sorted(path.name for path in profile.sessions_dir.iterdir() if path.is_dir() and path.name.startswith("session_")) if profile.sessions_dir.exists() else []
        missing = [name for name in selected_names if name not in set(local_sessions)]
        if missing:
            raise FileNotFoundError("Selected local curated sessions are missing: " + ", ".join(missing))
        print(f"Local curated sessions selected: {len(selected_names)}")
        return selected_names, selected_rows

    clear_rolling_replay_dirs()
    archive_root = ROLLING_REPLAY_ROOT / "_archives"
    archive_root.mkdir(parents=True, exist_ok=True)
    for row in selected_rows.itertuples(index=False):
        dataset_root = ROLLING_REPLAY_ROOT / str(row.dataset_id)
        dataset_root.mkdir(parents=True, exist_ok=True)
        archive_path = archive_root / f"{row.session_id}.tar.gz"
        print(f"Downloading {row.processed_path} -> {archive_path}")
        gcloud_storage_cp(row.processed_path, archive_path)
        extract_session_archive(archive_path, dataset_root, str(row.session_id))
    shutil.rmtree(archive_root, ignore_errors=True)
    print(f"Curated replay sessions downloaded: {len(selected_names)}")
    return selected_names, selected_rows


selected_sessions, selected_session_rows = download_rolling_sessions(profile, storage_profile)
print("Selected sessions:")
for name in selected_sessions:
    print(" -", name)
print(selected_session_rows[["session_id", "dataset_id", "split", "processed_path"]].to_string(index=False))


In [ ]:
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from branch2.navigation_config import BRANCH3_REPLAY_CONFIG, config_summary, validate_branch3_replay_config
from config.config import install_import_paths

notebook_nav_config = replace(
    BRANCH3_REPLAY_CONFIG,
    output_root=profile.output_dir,
    mmwave_cfg_path=profile.mmwave_cfg_path,
    model_pt=profile.branch1_checkpoint,
    unet=replace(
        BRANCH3_REPLAY_CONFIG.unet,
        enabled=True,
        checkpoint=profile.branch3_unet_checkpoint,
    ),
)

install_import_paths(
    ml_model_dir=notebook_nav_config.ml_model_dir,
    llm_dir=notebook_nav_config.llm_dir,
)
validate_branch3_replay_config(notebook_nav_config)
print(json.dumps(config_summary(notebook_nav_config), indent=2))

In [ ]:
def raw_session_csv_path(session_dir: Path) -> Path:
    canonical = session_dir / f"{session_dir.name}.csv"
    if canonical.exists():
        return canonical
    candidates = [
        p for p in sorted(session_dir.glob("*.csv"))
        if "labeled" not in p.name and "synchronized" not in p.name and not p.name.endswith("_timestamps.csv")
    ]
    return candidates[0] if candidates else canonical

required_paths = [
    profile.sessions_dir,
    profile.branch1_checkpoint,
    profile.branch3_unet_checkpoint,
    profile.mmwave_cfg_path,
    profile.extrinsics_json,
]
missing = [str(path) for path in required_paths if path is None or not Path(path).exists()]
if missing:
    raise FileNotFoundError("Preflight missing required paths:\n" + "\n".join(missing))

session_dirs = [profile.sessions_dir / name for name in selected_sessions]
missing_csv = [str(raw_session_csv_path(path)) for path in session_dirs if not raw_session_csv_path(path).exists()]
if missing_csv:
    raise FileNotFoundError("Selected sessions are missing raw radar CSVs:\n" + "\n".join(missing_csv))

preflight = {
    "run_id": profile.run_id,
    "sessions_dir": str(profile.sessions_dir),
    "selected_sessions": selected_sessions,
    "branch1_checkpoint": str(profile.branch1_checkpoint),
    "branch3_unet_checkpoint": str(profile.branch3_unet_checkpoint),
    "mmwave_cfg_path": str(profile.mmwave_cfg_path),
    "extrinsics_json": str(profile.extrinsics_json),
}
print(json.dumps(preflight, indent=2))

In [ ]:
replay_output = profile.output_dir / profile.run_id
replay_output.mkdir(parents=True, exist_ok=True)
summary_json = replay_output / "summary.json"
frame_predictions_csv = replay_output / "frame_predictions.csv"
point_predictions_csv = replay_output / "point_predictions.csv"

effective_session_selection = "sequential" if IN_COLAB else profile.session_selection
effective_max_sessions = len(selected_sessions) if IN_COLAB else profile.max_sessions

args = [
    "--sessions-dir", str(profile.sessions_dir),
    "--max-sessions", str(effective_max_sessions),
    "--session-selection", effective_session_selection,
    "--session-stride", str(profile.session_stride),
    "--session-seed", str(profile.session_seed),
    "--ui-style", profile.ui_style,
    "--output", str(replay_output),
    "--summary-json", str(summary_json),
    "--fps", str(profile.fps),
    "--width", str(profile.width),
    "--height", str(profile.height),
    "--extrinsics-json", str(profile.extrinsics_json),
    "--run-id", profile.run_id,
]
if profile.session_names:
    args.extend(["--session-names", profile.session_names])
if profile.no_video:
    args.append("--no-video")
if profile.no_color_video:
    args.append("--no-color-video")
if profile.debug_frame_snapshots:
    args.extend(["--debug-frame-snapshots", "--debug-frame-snapshot-stride", str(profile.debug_frame_snapshot_stride)])
if profile.export_frame_predictions:
    args.extend(["--export-frame-predictions", str(frame_predictions_csv)])
if profile.export_point_predictions:
    args.extend(["--export-point-predictions", str(point_predictions_csv)])

import sim_eval.branch3_replay_simulation as replay

# The replay script intentionally keeps navigation behavior out of CLI flags.
# Inject the notebook profile into its module-level canonical config before main().
replay.BRANCH3_REPLAY_CONFIG = notebook_nav_config
replay.REPLAY_NAVIGATION_CONFIG = notebook_nav_config
replay.DATA_DIR = profile.sessions_dir
replay.DEFAULT_EXTRINSICS_JSON = profile.extrinsics_json
replay.MIN_VALID_Z = notebook_nav_config.min_valid_z
replay.MIN_RANGE_M = notebook_nav_config.min_range_m
replay.MAX_RANGE_M = notebook_nav_config.max_range_m
replay._REPLAY_MODEL_CONTEXT = None
replay._REPLAY_MODEL_CONTEXT_KEY = None
replay._REPLAY_UNET_CONTEXT = None
replay._REPLAY_UNET_CONTEXT_KEY = None

old_argv = sys.argv[:]
try:
    sys.argv = ["branch3_replay_simulation.py", *args]
    replay.main()
finally:
    sys.argv = old_argv

print("Replay output:", replay_output)
print("Summary JSON:", summary_json)

In [ ]:
if profile.run_oneformer_eval:
    eval_out = replay_output / "oneformer_decision_eval"
    cmd = [
        sys.executable,
        str(CODE_ROOT / "sim_eval" / "evaluate_replay_decisions_against_oneformer.py"),
        "--frame-preds", str(frame_predictions_csv),
        "--processing-root", str(profile.processing_root),
        "--out-dir", str(eval_out),
        "--extrinsics-json", str(profile.extrinsics_json),
    ]
    if profile.sync_root is not None:
        cmd.extend(["--sync-root", str(profile.sync_root)])
    run(cmd, cwd=CODE_ROOT)
else:
    print("OneFormer decision evaluation disabled by profile.")

In [ ]:
if profile.run_overlay_diagnostics:
    overlay_out = replay_output / "seg_radar_overlays"
    cmd = [
        sys.executable,
        str(CODE_ROOT / "sim_eval" / "visualize_seg_radar_overlay.py"),
        "--processing-root", str(profile.processing_root),
        "--out-dir", str(overlay_out),
        "--with-depth",
        "--show-all-points",
        "--extrinsics", str(profile.extrinsics_json),
    ]
    run(cmd, cwd=CODE_ROOT)
else:
    print("Overlay diagnostics disabled by profile.")

In [ ]:
manifest = {
    "run_id": profile.run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "code_root": str(CODE_ROOT),
    "gcs_code_root": storage_profile.gcs_code_root,
    "gcs_catalog_uri": storage_profile.gcs_catalog_uri,
    "selected_sessions": selected_sessions,
    "selected_catalog_rows": selected_session_rows[["session_id", "dataset_id", "split", "processed_path"]].to_dict(orient="records"),
    "profile_json": str(profile_path),
    "output_dir": str(replay_output),
    "summary_json": str(summary_json),
    "frame_predictions_csv": str(frame_predictions_csv) if frame_predictions_csv.exists() else None,
    "point_predictions_csv": str(point_predictions_csv) if point_predictions_csv.exists() else None,
    "navigation_config_summary": config_summary(notebook_nav_config),
    "promote_to_published": profile.promote_to_published,
}
manifest_path = replay_output / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest:", manifest_path)

if IN_COLAB:
    gcs_run_out = f"{storage_profile.gcs_output_root.rstrip('/')}/{profile.run_id}"
    gcloud_storage_rsync(replay_output, gcs_run_out, delete=True)
    print("Uploaded:", gcs_run_out)
    if profile.promote_to_published:
        gcs_publish_out = f"{storage_profile.gcs_published_output_root.rstrip('/')}/{profile.run_id}"
        gcloud_storage_rsync(replay_output, gcs_publish_out, delete=True)
        print("Published:", gcs_publish_out)
else:
    print("Local runtime: upload skipped.")


In [ ]:
print("Replay artifacts")
for path in sorted(replay_output.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".mp4", ".json", ".csv", ".png"}:
        print(path.relative_to(replay_output))

if summary_json.exists():
    summary = json.loads(summary_json.read_text(encoding="utf-8"))
    compact = {
        "sessions": summary.get("sessions"),
        "frames": summary.get("frames"),
        "action_counts": summary.get("action_counts", summary.get("counts")),
        "branch3_available_frames": summary.get("branch3_available_frames"),
        "unet_disagree_frames": summary.get("unet_disagree_frames"),
        "novel_occupancy_frames": summary.get("novel_occupancy_frames"),
        "novel_free_space_frames": summary.get("novel_free_space_frames"),
    }
    print(json.dumps(compact, indent=2))

if IN_COLAB:
    from IPython.display import HTML, display
    mp4s = sorted(replay_output.glob("*.mp4"))
    if mp4s:
        display(HTML(f'<video controls width="960" src="{mp4s[0]}"></video>'))